# Iris classifier: EDA, tuning, explainability, and monitoring

This notebook is the guided entry point for the reproducible project workflow. The modelling logic lives in `src/` and is run through scripts so that the same process works outside Jupyter too.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'scripts').exists(), 'Open this notebook from the notebooks folder or project root.'
PROJECT_ROOT

## 1. Profile the data

The EDA checks completeness and duplicates, confirms class balance, profiles feature distributions by species, identifies IQR outlier flags without silently deleting records, and visualizes correlation and separability. `Id` is deliberately excluded from predictors because it is only a record key.

In [ ]:
subprocess.run([sys.executable, 'scripts/profile_data.py'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, 'scripts/export_notebook_figures.py'], cwd=PROJECT_ROOT, check=True)
display(Markdown((PROJECT_ROOT / 'reports' / 'data_profile.md').read_text(encoding='utf-8')))
Image(filename=PROJECT_ROOT / 'reports' / 'figures' / 'pairwise_relationships.png')

## 2. Tune and compare models

A stratified 80/20 train-test split protects a final hold-out set. Logistic Regression, KNN, Random Forest, and SVM pipelines are tuned with stratified 5-fold cross-validation only on training data. Macro F1 selects the winner, which gives every iris species equal weight; accuracy, balanced accuracy, and log loss are captured alongside it.

In [ ]:
subprocess.run([sys.executable, 'scripts/train_compare.py'], cwd=PROJECT_ROOT, check=True)
comparison = pd.read_csv(PROJECT_ROOT / 'reports' / 'model_comparison.csv')
display(comparison)
display(pd.read_json(PROJECT_ROOT / 'reports' / 'training_summary.json', typ='series'))
Image(filename=PROJECT_ROOT / 'reports' / 'figures' / 'confusion_matrix_best_model.png')

## 3. Explain the selected model with SHAP

SHAP shows which measurements had the greatest average influence on the selected model's predicted class probabilities. It is an explanation of the model—not a causal scientific claim. Permutation importance is also saved as a model-agnostic cross-check.

In [ ]:
display(pd.read_csv(PROJECT_ROOT / 'reports' / 'shap_feature_importance.csv'))
display(Markdown((PROJECT_ROOT / 'reports' / 'shap_summary.md').read_text(encoding='utf-8')))
Image(filename=PROJECT_ROOT / 'reports' / 'figures' / 'shap_feature_importance.png')

## 4. Score a batch and monitor it

Scoring writes predicted species and class probabilities. If the input also has `Species`, it records performance metrics. Monitoring compares incoming feature distributions against the training baseline with PSI, flags values outside the historical feature range, and tracks predicted-class mix and confidence. For the demonstration below, the original data is scored; use a genuinely new CSV for real monitoring.

In [ ]:
subprocess.run([sys.executable, 'scripts/score_model.py', '--input', 'Iris.csv'], cwd=PROJECT_ROOT, check=True)
display(pd.read_csv(PROJECT_ROOT / 'reports' / 'monitoring' / 'feature_drift.csv'))
display(pd.read_json(PROJECT_ROOT / 'reports' / 'monitoring' / 'monitoring_summary.json', typ='series'))